## 01 — Writing the LOD Files

We have a design. Now we execute it.

This notebook:

1. Defines the four LOD configurations
2. Implements the simplification pipeline
3. Writes four GeoJSON files to `data/lod/`
4. Reports what was produced

Run this notebook once. The output files are used by every module that follows.

## Setup

In [1]:
import json
import time
from pathlib import Path
from shapely.geometry import LineString

filename = "ne_10m_railroads.geojson"
candidates = [
    Path("../../data") / filename,
    Path("data") / filename,
    Path("Assignments_Completed/03-Data_Manager/data") / filename,
]

data_path = next((path for path in candidates if path.exists()), None)
if data_path is None:
    matches = list(Path.cwd().rglob(filename))
    if not matches:
        raise FileNotFoundError(f"Could not find {filename} from {Path.cwd()}")
    data_path = matches[0]

project_dir = data_path.parents[1]
output_dir = project_dir / "data" / "lod"
output_dir.mkdir(parents=True, exist_ok=True)

with open(data_path) as f:
    railroads = json.load(f)

features = railroads["features"]
print(f"Loaded {len(features):,} features")
print(f"Output directory: {output_dir.resolve()}")

Loaded 25,413 features
Output directory: /Users/josemarioomrqz/4543-5993-Spatial-Maps/Assignments_Completed/03-Data_Manager/data/lod


## LOD Configuration

Each level is defined by three parameters:
- `epsilon` — the simplification tolerance in degrees
- `max_scalerank` — maximum scalerank to include (`None` = include all)
- `zoom_range` — informational, used in the output properties

In [2]:
LOD_LEVELS = [
    {
        "name":          "coarse",
        "filename":      "railroads_coarse.geojson",
        "epsilon":       1.0,
        "max_scalerank": 4,
        "zoom_range":    "1-3",
    },
    {
        "name":          "medium",
        "filename":      "railroads_medium.geojson",
        "epsilon":       0.1,
        "max_scalerank": None,
        "zoom_range":    "4-6",
    },
    {
        "name":          "fine",
        "filename":      "railroads_fine.geojson",
        "epsilon":       0.01,
        "max_scalerank": None,
        "zoom_range":    "7-10",
    },
    {
        "name":          "extra_fine",
        "filename":      "railroads_extra_fine.geojson",
        "epsilon":       0.001,
        "max_scalerank": None,
        "zoom_range":    "11+",
    },
]

## The Simplification Function

One function handles a single feature at a given epsilon. It returns the simplified feature dict, or `None` if the feature collapses.

In [3]:
def simplify_feature(feature, epsilon):
    """
    Simplify one GeoJSON LineString feature using Shapely's Douglas-Peucker.
    Returns a new feature dict, or None if the result has fewer than 2 points.
    """
    coords = feature["geometry"]["coordinates"]

    simplified_geom = LineString(coords).simplify(epsilon, preserve_topology=False)
    simplified_coords = list(simplified_geom.coords)

    if len(simplified_coords) < 2:
        return None

    return {
        "type":       "Feature",
        "properties": feature["properties"],
        "geometry":   {
            "type":        "LineString",
            "coordinates": simplified_coords,
        },
    }

## The Pipeline

For each LOD level:
1. Filter by `max_scalerank` if set
2. Simplify each feature
3. Drop `None` results (collapsed features)
4. Write to a GeoJSON file

In [4]:
print(f"{'Level':<12} {'Epsilon':>8} {'In':>8} {'Out':>8} {'Dropped':>9} {'Size (MB)':>11} {'Time (s)':>10}")
print("-" * 72)

for level in LOD_LEVELS:
    t0 = time.perf_counter()

    # Step 1 — feature filter
    if level["max_scalerank"] is not None:
        candidates = [
            f for f in features
            if f["properties"]["scalerank"] <= level["max_scalerank"]
        ]
    else:
        candidates = features

    # Step 2 & 3 — simplify and drop collapsed
    simplified = []
    for f in candidates:
        result = simplify_feature(f, level["epsilon"])
        if result is not None:
            simplified.append(result)

    # Step 4 — write output
    collection = {"type": "FeatureCollection", "features": simplified}
    out_path = output_dir / level["filename"]

    with open(out_path, "w") as f:
        json.dump(collection, f)

    elapsed   = time.perf_counter() - t0
    size_mb   = out_path.stat().st_size / 1_000_000
    dropped   = len(candidates) - len(simplified)

    print(
        f"{level['name']:<12} {level['epsilon']:>8} "
        f"{len(candidates):>8,} {len(simplified):>8,} "
        f"{dropped:>9,} {size_mb:>10.2f} {elapsed:>10.2f}"
    )

print("-" * 72)
print(f"\nAll files written to: {output_dir.resolve()}")

Level         Epsilon       In      Out   Dropped   Size (MB)   Time (s)
------------------------------------------------------------------------
coarse            1.0    2,845    2,845         0       1.08       0.13


medium            0.1   25,413   25,413         0       9.66       1.22


fine             0.01   25,413   25,413         0      11.34       2.27


extra_fine      0.001   25,413   25,413         0      18.98       3.89
------------------------------------------------------------------------

All files written to: /Users/josemarioomrqz/4543-5993-Spatial-Maps/Assignments_Completed/03-Data_Manager/data/lod


## Verifying the Output Files

Let's confirm the files exist and are valid GeoJSON by loading one back.

In [5]:
print("Output files:")
for level in LOD_LEVELS:
    path = output_dir / level["filename"]
    size_mb = path.stat().st_size / 1_000_000
    print(f"  {level['filename']:<40} {size_mb:.2f} MB")

print()

# Spot-check: load the fine level and verify structure
with open(output_dir / "railroads_fine.geojson") as f:
    check = json.load(f)

print(f"Fine level spot-check:")
print(f"  type:          {check['type']}")
print(f"  feature count: {len(check['features']):,}")
print(f"  first feature geometry type: {check['features'][0]['geometry']['type']}")
print(f"  first feature coord count:   {len(check['features'][0]['geometry']['coordinates'])}")

Output files:
  railroads_coarse.geojson                 1.08 MB
  railroads_medium.geojson                 9.66 MB
  railroads_fine.geojson                   11.34 MB
  railroads_extra_fine.geojson             18.98 MB

Fine level spot-check:
  type:          FeatureCollection
  feature count: 25,413
  first feature geometry type: LineString
  first feature coord count:   2


## Exercise A

The coarse level applies a `scalerank <= 4` filter before simplification. Modify the pipeline (in a copy below) to run the coarse level **without** the scalerank filter.

Compare:
- How many features does the unfiltered coarse level contain?
- How much larger is the file?
- Is the scalerank filter worth keeping, or is geometry simplification alone sufficient?

Write your conclusion as a comment.

In [6]:
# Run coarse simplification without the scalerank filter
# Compare against the filtered version

coarse_level = next(level for level in LOD_LEVELS if level["name"] == "coarse")
filtered_path = output_dir / coarse_level["filename"]

with open(filtered_path) as f:
    filtered_coarse = json.load(f)

unfiltered_features = []
for feature in features:
    result = simplify_feature(feature, coarse_level["epsilon"])
    if result is not None:
        unfiltered_features.append(result)

unfiltered_coarse = {
    "type": "FeatureCollection",
    "features": unfiltered_features,
}

filtered_count = len(filtered_coarse["features"])
unfiltered_count = len(unfiltered_coarse["features"])

filtered_size_mb = filtered_path.stat().st_size / 1_000_000
unfiltered_size_mb = len(json.dumps(unfiltered_coarse).encode("utf-8")) / 1_000_000
size_difference_mb = unfiltered_size_mb - filtered_size_mb
percent_larger = (size_difference_mb / filtered_size_mb) * 100

print(f"Filtered coarse features:   {filtered_count:,}")
print(f"Unfiltered coarse features: {unfiltered_count:,}")
print(f"Filtered coarse size:       {filtered_size_mb:.2f} MB")
print(f"Unfiltered coarse size:     {unfiltered_size_mb:.2f} MB")
print(f"Size increase:              {size_difference_mb:.2f} MB ({percent_larger:.1f}% larger)")

# Conclusion: the scalerank filter is worth keeping for the coarse LOD level.
# Geometry simplification alone keeps all 25,413 features, so the map still has to draw every railroad segment.
# The scalerank filter reduces the coarse layer to the most important 2,845 features and keeps the low-zoom file much smaller.

Filtered coarse features:   2,845
Unfiltered coarse features: 25,413
Filtered coarse size:       1.08 MB
Unfiltered coarse size:     9.55 MB
Size increase:              8.47 MB (786.9% larger)


## Exercise B

The pipeline writes GeoJSON using `json.dump()`, which produces unformatted output (no indentation). This is intentional — indentation adds whitespace that inflates file size.

1. Write one of the LOD files again with `indent=2` to make it human-readable.
2. Compare the file size before and after.
3. By what percentage does indentation increase file size?

Do not keep the indented file — overwrite it with the compact version when you are done.

In [7]:
# Write a LOD file with indent=2, measure the size difference, then restore compact version

level = next(level for level in LOD_LEVELS if level["name"] == "medium")
path = output_dir / level["filename"]

with open(path) as f:
    data = json.load(f)

compact_size_mb = path.stat().st_size / 1_000_000

# Temporarily write a human-readable version.
with open(path, "w") as f:
    json.dump(data, f, indent=2)

indented_size_mb = path.stat().st_size / 1_000_000
increase_mb = indented_size_mb - compact_size_mb
percent_increase = (increase_mb / compact_size_mb) * 100

print(f"File tested:        {path.name}")
print(f"Compact size:       {compact_size_mb:.2f} MB")
print(f"Indented size:      {indented_size_mb:.2f} MB")
print(f"Increase:           {increase_mb:.2f} MB")
print(f"Percent increase:   {percent_increase:.1f}%")

# Restore the compact version so the LOD file stays optimized for the map pipeline.
with open(path, "w") as f:
    json.dump(data, f)

restored_size_mb = path.stat().st_size / 1_000_000
print(f"Restored size:      {restored_size_mb:.2f} MB")

# Conclusion: indentation makes GeoJSON easier for humans to read, but it increases file size
# without adding information. For map delivery, compact JSON is the better default.

File tested:        railroads_medium.geojson
Compact size:       9.66 MB
Indented size:      16.44 MB
Increase:           6.77 MB
Percent increase:   70.1%


Restored size:      9.66 MB


## Check Your Understanding

We use `shapely.simplify(epsilon, preserve_topology=False)`.

Shapely also has `preserve_topology=True`. Look up what that parameter does, then answer:

For a railroad LOD pipeline, should we use `True` or `False`? Why?

Write a 2–3 sentence answer.

`preserve_topology=True` makes Shapely avoid simplifications that would create invalid geometry, such as self-intersections. For this railroad LOD pipeline, I would keep `preserve_topology=False` because the data is made of LineStrings and the main goal is fast visual simplification for map rendering. Topology preservation matters more for polygons or shared boundaries, where invalid shapes would create serious display or analysis problems.

---

## Next

In [02 — Comparing the Levels](./02-Comparing_the_Levels.ipynb), we load all four output files and compare them visually on a map.